# 셋팅 #

필요한 설정을 위해 이 셀을 실행하세요!


In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
store_sales_time_series_forecasting_path = kagglehub.competition_download('store-sales-time-series-forecasting')
ryanholbrook_ts_course_data_path = kagglehub.dataset_download('ryanholbrook/ts-course-data')

print('Data source import complete.')


In [ ]:
import os
from pathlib import Path

# Define the target directory where learntools expects to find the data
input_base_path = Path('/input')
input_base_path.mkdir(parents=True, exist_ok=True)

# Create a symbolic link for the store-sales-time-series-forecasting competition data
symlink_target_comp = input_base_path / 'store-sales-time-series-forecasting'
symlink_source_comp = store_sales_time_series_forecasting_path

if symlink_target_comp.exists() or symlink_target_comp.is_symlink():
    if symlink_target_comp.is_symlink():
        os.unlink(symlink_target_comp)
    else:
        # If it's a directory, remove it to replace with symlink
        import shutil
        shutil.rmtree(symlink_target_comp)

os.symlink(symlink_source_comp, symlink_target_comp)
print(f"Created symlink: {symlink_target_comp} -> {symlink_source_comp}")

# Create a symbolic link for the ryanholbrook/ts-course-data dataset
symlink_target_data = input_base_path / 'ts-course-data'
symlink_source_data = ryanholbrook_ts_course_data_path

if symlink_target_data.exists() or symlink_target_data.is_symlink():
    if symlink_target_data.is_symlink():
        os.unlink(symlink_target_data)
    else:
        import shutil
        shutil.rmtree(symlink_target_data)

os.symlink(symlink_source_data, symlink_target_data)
print(f"Created symlink: {symlink_target_data} -> {symlink_source_data}")


In [ ]:
!git clone https://github.com/Kaggle/learntools.git
!mv learntools learntools_dir
!mv learntools_dir/learntools learntools

# 소개 #

선형 회귀는 추세를 바깥으로까지(extrapolation) 이어 그리는 데 뛰어나지만 피처 간 상호작용을 학습하지는 못합니다. 반대로 XGBoost는 복잡한 상호작용을 훌륭하게 학습하지만 추세를 외삽하지는 못합니다. 이번 강의에서는 서로를 보완하는 학습 알고리즘을 결합해 한쪽의 강점으로 다른 쪽의 약점을 보완하는 “하이브리드” 예측기를 만드는 방법을 배웁니다.

# 구성 요소와 잔차 #

효과적인 하이브리드를 설계하려면 시계열이 어떻게 구성되는지 더 잘 이해해야 합니다. 지금까지 우리는 추세(trend), 계절성(season), 주기(cycle)라는 세 가지 의존 패턴을 살펴봤습니다. 많은 시계열은 이 세 가지 구성 요소와 예측할 수 없는 무작위 *오차(error)* 를 더한 가법 모델로 근사할 수 있습니다.

```
series = trend + seasons + cycles + error
```

이 모델에 등장하는 항을 각각 시계열의 **구성 요소(component)** 라고 부르겠습니다.

모델의 **잔차(residual)** 는 모델이 학습한 타깃과 모델이 만든 예측의 차이입니다. 즉, 실제 곡선과 적합된 곡선 사이의 차이입니다. 잔차를 특정 피처에 대해 플롯하면 타깃에서 그 피처가 설명하지 못하고 남긴 부분, 다시 말해 모델이 배우지 못한 부분을 볼 수 있습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/mIeeaBD.png" width=700, alt="">
<figcaption style="textalign: center; font-style: italic"><center>타깃 시리즈와 예측(파란색)의 차이가 잔차 시리즈입니다.
</center></figcaption>
</figure>

위 그림의 왼쪽은 *Tunnel Traffic* 시리즈 일부와 3강에서 만든 추세+계절 곡선입니다. 적합된 곡선을 빼면 오른쪽처럼 잔차만 남습니다. 잔차에는 추세-계절 모델이 배우지 못한 *Tunnel Traffic* 의 모든 정보가 들어 있습니다.

시계열의 구성 요소를 학습하는 과정을 다음과 같이 반복적인 단계로 상상해 볼 수 있습니다. 먼저 추세를 학습해 시리즈에서 뺍니다. 그다음 디트렌딩된 잔차에서 계절성을 학습해 또 뺍니다. 이어서 주기를 학습해 제거하고 나면, 결국 예측 불가능한 오차만 남습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/XGJuheO.png" width=700, alt="">
<figcaption style="textalign: center; font-style: italic"><center><em>Mauna Loa CO2</em> 시리즈의 구성 요소를 단계별로 학습하기. 파란 곡선을 빼면 다음 단계의 시리즈가 됩니다.
</center></figcaption>
</figure>

학습한 구성 요소를 모두 더하면 완전한 모델을 얻습니다. 추세·계절·주기를 모두 모델링하는 피처를 제공한다면 선형 회귀가 본질적으로 하는 일이 바로 이것입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/HZEhuHF.png" width=600, alt="">
<figcaption style="textalign: center; font-style: italic"><center>학습한 구성 요소를 더해 완전한 모델을 만든 모습.
</center></figcaption>
</figure>

# 잔차를 활용한 하이브리드 예측 #

앞선 강의에서는 하나의 알고리즘(선형 회귀)으로 모든 구성 요소를 한꺼번에 학습했습니다. 하지만 어떤 구성 요소에는 한 알고리즘을, 다른 구성 요소에는 또 다른 알고리즘을 사용할 수도 있습니다. 이렇게 하면 각 구성 요소를 가장 잘 학습하는 알고리즘을 선택할 수 있습니다. 구체적으로는 한 알고리즘으로 원래 시리즈를 적합한 뒤, 다른 알고리즘으로 잔차 시리즈를 적합합니다.

```
# 1. 첫 번째 모델을 학습하고 예측
model_1.fit(X_train_1, y_train)
y_pred_1 = model_1.predict(X_train)

# 2. 잔차에 대해 두 번째 모델을 학습하고 예측
model_2.fit(X_train_2, y_train - y_pred_1)
y_pred_2 = model_2.predict(X_train_2)

# 3. 두 예측을 더해 최종 예측 생성
y_pred = y_pred_1 + y_pred_2
```

각 모델에 어떤 역할을 맡길지에 따라 서로 다른 피처 집합(`X_train_1`, `X_train_2`)을 사용하는 편이 일반적입니다. 예를 들어 첫 번째 모델이 추세를 학습했다면, 두 번째 모델에는 굳이 추세 피처를 넣지 않아도 됩니다.

두 개 이상의 모델을 중첩해 사용하는 것도 가능하지만, 실무에서는 그다지 도움이 되지 않는 경우가 많습니다. 실제로 가장 흔한 하이브리드 구성은 지금 설명한 형태, 즉 단순(대개 선형) 모델 다음에 복잡한 비선형 모델(GBDT나 딥러닝 등)을 두는 방식입니다. 앞선 단순 모델은 뒷부분의 강력한 알고리즘을 돕기 위한 보조 역할에 가깝습니다.

### 하이브리드 설계하기

이번 강의에서 소개한 방법 말고도 머신러닝 모델을 결합하는 다양한 방식이 있습니다. 다만 모델을 성공적으로 결합하려면 각 알고리즘이 어떻게 작동하는지 좀 더 깊이 들여다볼 필요가 있습니다.

회귀 알고리즘이 예측을 만드는 방식은 크고 작게 두 가지로 나눌 수 있습니다. 하나는 *피처* 를 변환하는 방식이고, 다른 하나는 *타깃* 을 변환하는 방식입니다. 피처 변환 알고리즘은 피처를 입력으로 받아 수학적 함수를 학습한 뒤, 이를 조합·변형해 학습 세트의 타깃 값을 맞춥니다. 선형 회귀와 신경망이 여기에 해당합니다.

타깃 변환 알고리즘은 피처를 이용해 타깃 값을 여러 그룹으로 나눈 다음, 각 그룹의 평균으로 예측합니다. 피처 집합은 어떤 그룹을 평균낼지를 가리키는 역할을 합니다. 결정 트리와 최근접 이웃(KNN)이 이 방식입니다.

중요한 차이는 다음과 같습니다. 피처 변환 알고리즘은 적절한 피처만 제공된다면 학습 세트 범위를 넘어서는 값을 **외삽** 할 수 있지만, 타깃 변환 알고리즘의 예측은 항상 학습 세트의 범위 안에 머뭅니다. 시간 더미가 계속 증가하면 선형 회귀는 추세선을 계속 그릴 수 있지만, 같은 시간 더미를 본 결정 트리는 학습 데이터의 마지막 값에서 멈춘 추세를 앞으로도 그대로 복제할 뿐입니다. 즉, *결정 트리는 추세를 외삽하지 못합니다.* 결정 트리를 여러 개 모은 랜덤 포레스트나 그래디언트 부스팅 계열(예: XGBoost)도 마찬가지입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/ZZtfuFJ.png" width=600, alt="">
<figcaption style="textalign: center; font-style: italic"><center>결정 트리는 학습 구간 밖으로 추세를 외삽하지 못합니다.
</center></figcaption>
</figure>

이번 강의에서 소개하는 하이브리드 설계는 바로 이 차이를 활용합니다. 선형 회귀로 추세를 외삽하고, 그 예측을 타깃에서 빼서 추세를 제거한 뒤, 남은 잔차를 XGBoost로 학습합니다. 만약 신경망(피처 변환 알고리즘)을 하이브리드로 사용한다면, 다른 모델의 예측을 피처로 추가해 신경망이 그 값을 참고하도록 만들 수 있습니다. 잔차를 학습하는 방식은 그래디언트 부스팅 알고리즘 내부에서 쓰는 것과 동일하기 때문에 이런 하이브리드를 **부스티드(boosted)** 하이브리드라고 부르고, 예측을 피처로 넘겨주는 방식은 “스태킹(stacking)”이라고 부르므로 **스택드(stacked)** 하이브리드라고 부르겠습니다.

<blockquote style="margin-right:auto; margin-left:auto; background-color: #ebf9ff; padding: 1em; margin:24px;">
<strong>Kaggle 대회 우승 하이브리드 예시</strong>
    <p>영감을 위해 과거 대회에서 높은 성적을 거둔 하이브리드 솔루션을 살펴보세요.</p>
<ul>
    <li><a href="https://www.kaggle.com/c/walmart-recruiting-store-sales-forecasting/discussion/8125">STL + 지수 평활 부스팅</a> - Walmart Recruiting - Store Sales Forecasting</li>
    <li><a href="https://www.kaggle.com/c/rossmann-store-sales/discussion/17896">ARIMA·지수 평활 + GBDT 부스팅</a> - Rossmann Store Sales</li>
    <li><a href="https://www.kaggle.com/c/web-traffic-time-series-forecasting/discussion/39395">스택드·부스티드 하이브리드 앙상블</a> - Web Traffic Time Series Forecasting</li>
    <li><a href="https://github.com/Mcompetitions/M4-methods/blob/slaweks_ES-RNN/118%20-%20slaweks17/ES_RNN_SlawekSmyl.pdf">지수 평활 + LSTM 스태킹</a> - M4 (Kaggle 외 대회)</li>
</ul>
</blockquote>

# 예제 - US Retail Sales #

[*US Retail Sales*](https://www.census.gov/retail/index.html) 데이터셋은 1992년부터 2019년까지 주요 소매 업종의 월별 매출을 담고 있습니다(미국 인구조사국 수집). 목표는 과거 판매량을 이용해 2016~2019년 매출을 예측하는 것입니다. 선형 회귀 + XGBoost 하이브리드를 만드는 동시에, XGBoost가 사용할 수 있도록 시계열 데이터를 구성하는 방법도 살펴보겠습니다.


In [ ]:
from pathlib import Path
from warnings import simplefilter

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from xgboost import XGBRegressor


simplefilter("ignore")

# Matplotlib 기본 설정
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc(
    "figure",
    autolayout=True,
    figsize=(11, 4),
    titlesize=18,
    titleweight='bold',
)
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=16,
    titlepad=10,
)
plot_params = dict(
    color="0.75",
    style=".-",
    markeredgecolor="0.25",
    markerfacecolor="0.25",
)

data_dir = Path("../input/ts-course-data/")
industries = ["BuildingMaterials", "FoodAndBeverage"]
retail = pd.read_csv(
    data_dir / "us-retail-sales.csv",
    usecols=['Month'] + industries,
    parse_dates=['Month'],
    index_col='Month',
).to_period('D').reindex(columns=industries)
retail = pd.concat({'Sales': retail}, names=[None, 'Industries'], axis=1)

retail.head()


먼저 선형 회귀 모델로 각 시계열의 추세를 학습하겠습니다. 시연을 위해 2차(제곱) 추세를 사용하겠습니다. (코드는 이전 강의와 거의 같습니다.) 완벽하게 들어맞지는 않지만 현재 목적에는 충분합니다.


In [ ]:
y = retail.copy()

# 추세 피처 생성
dp = DeterministicProcess(
    index=y.index,  # 학습 데이터의 날짜 인덱스
    constant=True,  # 절편
    order=2,        # 이차 추세
    drop=True,      # 다중공선성을 피하기 위해 항 제거
)
X = dp.in_sample()  # 학습 데이터용 피처

# 2016~2019년 데이터로 테스트합니다. 나중을 대비해
# 데이터프레임 대신 날짜 인덱스를 나눠 두겠습니다.
idx_train, idx_test = train_test_split(
    y.index, test_size=12 * 4, shuffle=False,
)
X_train, X_test = X.loc[idx_train, :], X.loc[idx_test, :]
y_train, y_test = y.loc[idx_train], y.loc[idx_test]

# 추세 모델 학습
model = LinearRegression(fit_intercept=False)
model.fit(X_train, y_train)

# 예측 생성
y_fit = pd.DataFrame(
    model.predict(X_train),
    index=y_train.index,
    columns=y_train.columns,
)
y_pred = pd.DataFrame(
    model.predict(X_test),
    index=y_test.index,
    columns=y_test.columns,
)

# 플롯
axs = y_train.plot(color='0.25', subplots=True, sharex=True)
axs = y_test.plot(color='0.25', subplots=True, sharex=True, ax=axs)
axs = y_fit.plot(color='C0', subplots=True, sharex=True, ax=axs)
axs = y_pred.plot(color='C3', subplots=True, sharex=True, ax=axs)
for ax in axs: ax.legend([])
_ = plt.suptitle("Trends")


선형 회귀는 다중 출력 회귀가 가능하지만 XGBoost는 그렇지 않습니다. 따라서 여러 시리즈를 동시에 예측하려면, 열마다 시계열이 있는 *와이드 형식* 데이터를 행을 따라 시리즈를 분류하는 *롱 형식* 데이터로 변환해야 합니다.


In [ ]:
# `stack` 메서드는 열 레이블을 행 레이블로 옮겨
# 와이드 형식을 롱 형식으로 피벗합니다.
X = retail.stack()  # 와이드 -> 롱 변환
display(X.head())
y = X.pop('Sales')  # 타깃 시리즈 추출


XGBoost가 두 시계열을 구분해 학습하도록 `'Industries'` 행 라벨을 범주형 피처로 바꾸고 라벨 인코딩을 적용하겠습니다. 또한 시간 인덱스에서 월 번호를 추출해 연간 계절성 피처를 추가합니다.


In [ ]:
# 행 레이블을 범주형 피처 열로 바꾸고 라벨 인코딩 적용
X = X.reset_index('Industries')
# 'Industries' 피처 라벨 인코딩
for colname in X.select_dtypes(["object", "category"]):
    X[colname], _ = X[colname].factorize()

# 연간 계절성 인코딩
X["Month"] = X.index.month  # 값은 1, 2, ..., 12

# 데이터 분할
X_train, X_test = X.loc[idx_train, :], X.loc[idx_test, :]
y_train, y_test = y.loc[idx_train], y.loc[idx_test]


이제 앞에서 만든 추세 예측을 롱 형식으로 변환한 뒤 원래 시리즈에서 빼겠습니다. 그러면 추세가 제거된 잔차 시리즈가 생기고, 이를 XGBoost가 학습하게 됩니다.


In [ ]:
# 추세 예측을 롱 형식으로 변환하고 시리즈 형태로 변경
y_fit = y_fit.stack().squeeze()    # 학습 구간 추세
y_pred = y_pred.stack().squeeze()  # 테스트 구간 추세

# 학습 구간에서 잔차(디트렌딩된 시리즈) 생성
y_resid = y_train - y_fit

# 잔차에 대해 XGBoost 학습
xgb = XGBRegressor()
xgb.fit(X_train, y_resid)

# 예측된 잔차를 추세 예측에 더하기
y_fit_boosted = xgb.predict(X_train) + y_fit
y_pred_boosted = xgb.predict(X_test) + y_pred


적합 결과가 꽤 좋아 보이지만, XGBoost가 학습한 추세는 선형 회귀가 학습한 추세만큼만 좋다는 점에 유의하세요. 특히 `'BuildingMaterials'` 시리즈에서 추세를 제대로 맞추지 못했는데, XGBoost가 이를 보완하지는 못했습니다.


In [ ]:

axs = y_train.unstack(['Industries']).plot(
    color='0.25', figsize=(11, 5), subplots=True, sharex=True,
    title=['BuildingMaterials', 'FoodAndBeverage'],
)
axs = y_test.unstack(['Industries']).plot(
    color='0.25', subplots=True, sharex=True, ax=axs,
)
axs = y_fit_boosted.unstack(['Industries']).plot(
    color='C0', subplots=True, sharex=True, ax=axs,
)
axs = y_pred_boosted.unstack(['Industries']).plot(
    color='C3', subplots=True, sharex=True, ax=axs,
)
for ax in axs: ax.legend([])

# 소개 #

환경을 모두 준비하려면 이 셀을 실행하세요!



In [ ]:
# 피드백 시스템 설정
from learntools.core import binder
binder.bind(globals())
from learntools.time_series.ex5 import *

# 노트북 환경 설정
from pathlib import Path
from learntools.time_series.style import *  # 플롯 스타일 설정

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.deterministic import DeterministicProcess
from xgboost import XGBRegressor


comp_dir = Path('../input/store-sales-time-series-forecasting')
data_dir = Path("../input/ts-course-data")

store_sales = pd.read_csv(
    comp_dir / 'train.csv',
    usecols=['store_nbr', 'family', 'date', 'sales', 'onpromotion'],
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'sales': 'float32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)
store_sales['date'] = store_sales.date.dt.to_period('D')
store_sales = store_sales.set_index(['store_nbr', 'family', 'date']).sort_index()

family_sales = (
    store_sales
    .groupby(['family', 'date'])
    .mean()
    .unstack('family')
    .loc['2017']
)


-------------------------------------------------------------------------------

다음 두 문제에서는 새로운 파이썬 클래스를 구현해 *Store Sales* 데이터셋용 부스티드 하이브리드를 만들겠습니다. 아래 셀을 실행하면 초기 클래스 정의가 만들어집니다. scikit-learn 스타일의 인터페이스를 제공하도록 `fit`과 `predict` 메서드를 직접 추가할 것입니다.


In [ ]:
# fit과 predict 메서드를 추가할 최소한의 클래스
class BoostedHybrid:
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2
        self.y_columns = None  # fit 메서드에서 받은 열 이름을 저장


# 1) 부스티드 하이브리드의 fit 메서드 정의하기

`BoostedHybrid` 클래스의 `fit` 메서드를 완성하세요. 필요하다면 튜토리얼의 **Residual을 활용한 하이브리드 예측** 섹션에 있는 1·2단계를 참고하세요.


In [ ]:
def fit(self, X_1, X_2, y):
    # 직접 작성하세요: self.model_1 학습
    ____

    y_fit = pd.DataFrame(
        # 직접 작성하세요: self.model_1으로 예측
        ____ ,
        index=X_1.index, columns=y.columns,
    )

    # 직접 작성하세요: 잔차 계산
    y_resid = ____
    y_resid = y_resid.stack().squeeze() # 와이드 -> 롱

    # 직접 작성하세요: 잔차에 대해 self.model_2 학습
    self.model_2.fit(____, ____)

    # predict 메서드를 위한 열 이름 저장
    self.y_columns = y.columns
    # 문제 채점을 위한 데이터 저장
    self.y_fit = y_fit
    self.y_resid = y_resid


# 클래스로 메서드 추가
BoostedHybrid.fit = fit



In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_1.hint()
#q_1.solution()


-------------------------------------------------------------------------------

# 2) 부스티드 하이브리드의 predict 메서드 정의하기

이제 `BoostedHybrid` 클래스의 `predict` 메서드를 정의하세요. 필요하면 튜토리얼의 3단계를 다시 확인하세요.


In [ ]:
def predict(self, X_1, X_2):
    y_pred = pd.DataFrame(
        # 직접 작성하세요: self.model_1으로 예측
        ____ ,
        index=X_1.index, columns=self.y_columns,
    )
    y_pred = y_pred.stack().squeeze()  # 와이드 -> 롱

    # 직접 작성하세요: self.model_2 예측을 y_pred에 더하기
    y_pred += ____

    return y_pred.unstack()  # 롱 -> 와이드


# 클래스로 메서드 추가
BoostedHybrid.predict = predict



In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_2.hint()
#q_2.solution()


-------------------------------------------------------------------------------

이제 방금 만든 `BoostedHybrid` 클래스를 사용해 *Store Sales* 데이터용 모델을 만들 준비가 되었습니다. 다음 셀을 실행하면 학습용 데이터가 준비됩니다.


In [ ]:
# 타깃 시리즈
y = family_sales.loc[:, 'sales']


# X_1: 선형 회귀용 피처
dp = DeterministicProcess(index=y.index, order=1)
X_1 = dp.in_sample()


# X_2: XGBoost용 피처
X_2 = family_sales.drop('sales', axis=1).stack()  # onpromotion 피처

# 'family' 라벨 인코딩
le = LabelEncoder()  # sklearn.preprocessing
X_2 = X_2.reset_index('family')
X_2['family'] = le.fit_transform(X_2['family'])

# 계절성 인코딩
X_2["day"] = X_2.index.day  # 값은 일(day) 번호


# 3) 부스티드 하이브리드 학습하기

`LinearRegression()`와 `XGBRegressor()` 인스턴스를 사용해 `BoostedHybrid` 클래스를 초기화하고 하이브리드 모델을 만드세요.


In [ ]:
# 직접 작성하세요: LinearRegression + XGBRegressor 하이브리드 만들기
model = ____

# 직접 작성하세요: 학습 및 예측
model.fit(____, ____, ____)
y_pred = ____

y_pred = y_pred.clip(0.0)




In [ ]:
# 아래 줄은 힌트나 해설 코드를 보여 줍니다
#q_3.hint()
#q_3.solution()


-------------------------------------------------------------------------------

문제에 따라 방금 만든 선형 회귀 + XGBoost 하이브리드 대신 다른 조합을 쓰고 싶을 수도 있습니다. 다음 셀에서는 scikit-learn의 다른 알고리즘을 시도해 볼 수 있습니다.


In [ ]:
# 모델 1(추세)
from sklearn.linear_model import ElasticNet, Lasso, Ridge

# 모델 2
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

# 부스티드 하이브리드

# 위 알고리즘을 다양하게 조합해 보세요
model = BoostedHybrid(
    model_1=Ridge(),
    model_2=KNeighborsRegressor(),
)


이것은 단지 몇 가지 예시일 뿐입니다. scikit-learn [사용자 가이드](https://scikit-learn.org/stable/supervised_learning.html)에서 마음에 드는 알고리즘을 더 찾아볼 수도 있습니다. 이 셀의 코드는 하이브리드가 만드는 예측을 시각화합니다.


In [ ]:
y_train, y_valid = y.loc[:pd.Period("2017-07-01", freq='D')], y.loc[pd.Period("2017-07-02", freq='D'):]
X1_train, X1_valid = X_1.loc[:pd.Period("2017-07-01", freq='D')], X_1.loc[pd.Period("2017-07-02", freq='D'):]
X2_train, X2_valid = X_2.loc[:pd.Period("2017-07-01", freq='D')], X_2.loc[pd.Period("2017-07-02", freq='D'):]

# 위 알고리즘 중 일부는 (표준화처럼) 특정 전처리를 하면 더 잘 동작합니다.
# 여기서는 단순한 데모만 진행합니다.
model.fit(X1_train, X2_train, y_train)
y_fit = model.predict(X1_train, X2_train).clip(0.0)
y_pred = model.predict(X1_valid, X2_valid).clip(0.0)

families = y.columns[0:6]
axs = y.loc(axis=1)[families].plot(
    subplots=True, sharex=True, figsize=(11, 9), **plot_params, alpha=0.5,
)
_ = y_fit.loc(axis=1)[families].plot(subplots=True, sharex=True, color='C0', ax=axs)
_ = y_pred.loc(axis=1)[families].plot(subplots=True, sharex=True, color='C3', ax=axs)
for ax, family in zip(axs, families):
    ax.legend([])
    ax.set_ylabel(family)